In [ ]:
import sys
import os

# Add paths to import from long_form_factuality
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
lff_root = os.path.join(project_root, "long_form_factuality")
for path in [project_root, lff_root]:
    if path not in sys.path:
        sys.path.insert(0, path)


from vllm_wrapper import VLLMRaterModel
from eval.safe.rate_atomic_fact import check_atomic_fact

In [4]:
import json
atomic_fact_data = []
# In Jupyter notebooks, use getcwd() instead of __file__
path = os.path.join(os.getcwd(), "data_for_git", "atomic_facts.jsonl")
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        atomic_fact_data.append(json.loads(line))

completed_facts = []
path_to_completed = os.path.join(os.getcwd(), "data_for_git", "rated_facts.jsonl")
with open(path_to_completed, "r", encoding="utf-8") as f:
    for line in f:
        completed_facts.append(json.loads(line))

print("number of completed facts: ", len(completed_facts))
print(completed_facts[0]["id"])


# check how many unique ids are in atomic_fact_data
atomic_ids = [fact["id"] for fact in atomic_fact_data]
atomic_unique_ids = set(atomic_ids)
print("number of unique ids in atomic_fact_data: ", len(atomic_unique_ids))



# Check for duplicate IDs in completed_facts
ids = [fact["id"] for fact in completed_facts]
unique_ids = set(ids)
duplicate_ids = [id for id in unique_ids if ids.count(id) > 1]

print(len(duplicate_ids))

# filter atomic_fact_data to only include facts that are not in completed_facts
# Count total unique facts (similar to facts_completed but without duplicates)
# Convert dicts to JSON strings for hashing (since dicts are not hashable)
unique_completed_facts = len(set(json.dumps(fact, sort_keys=True) for query in completed_facts for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]))
print("unique_completed_facts: ", unique_completed_facts)
before = len(atomic_fact_data)
facts_before = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])
completed_ids_set = set(ids)  # Use the IDs we already extracted above
atomic_fact_data = [fact for fact in atomic_fact_data if fact["id"] not in completed_ids_set]
after = len(atomic_fact_data)
filtered_out = before - after
print(f"Before: {before}, After: {after}, Filtered out: {filtered_out}")


number of completed facts:  3194
be3ff6b1-3f3e-4d3c-bcef-62473da8c672
number of unique ids in atomic_fact_data:  3413
85
unique_completed_facts:  97108
Before: 3413, After: 318, Filtered out: 3095


In [5]:
total_facts = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])
print(total_facts)
print(len(atomic_fact_data))
atomic_fact_data[0]["results"]["all_atomic_facts"][0]["atomic_facts"]

facts_completed = len([fact for query in completed_facts for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])
total_facts = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])

print(unique_completed_facts)
print(facts_completed)
print(total_facts)
print(facts_before)

17835
318
97108
107277
17835
124643


In [ ]:
llm = VLLMRaterModel()
import threading
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import json

MAX_WORKERS = 400


# Thread-safe list to collect results
results = []
results_lock = threading.Lock()
error_log = []
error_log_lock = threading.Lock()
pbar_lock = threading.Lock()

def worker(full_dict, pbar):
    try:
        # Extract prompt logic...
        bio_person = full_dict["prompt"].split("Tell me a bio of ")[1]
        
        all_atomic_facts = full_dict["results"]["all_atomic_facts"]
        for sentence_facts in all_atomic_facts:
            for fact_index, fact in enumerate(sentence_facts["atomic_facts"]):
                # 'llm' is captured from outer scope or passed via partial
                rating = check_atomic_fact(fact, bio_person, llm, max_steps=2)[0].answer
                sentence_facts["atomic_facts"][fact_index] = {"fact": fact, "rating": rating}          
                with pbar_lock:
                    pbar.update(1)
        return ("success", full_dict)
    except Exception as e:
        return ("error", f"Error processing response {full_dict.get('id', '?')}: {e}")

first_n = None
facts_completed = len([fact for query in completed_facts for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])
total_facts = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])
current_progress = len(completed_facts)

# Define output file for incremental saving
output_file = os.path.join(os.getcwd(), "data_for_git", "rated_facts.jsonl")

# 2. Run wth Executor
tasks = atomic_fact_data
# Create both progress bars with position parameter to display them separately
with tqdm(total=facts_before, desc="Rating Facts", position=0, leave=True) as task_bar:
    task_bar.update(unique_completed_facts)
    with tqdm(total=before, desc="Completed Responses", position=1, leave=True) as pbar:
        pbar.update(filtered_out)
        tqdm.write(f"Starting from {filtered_out}/{before} ({filtered_out/before * 100}%)")
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # Submit all tasks
            futures = [executor.submit(worker, task, task_bar) for task in tasks]
            
            # Process results as they complete
            for future in as_completed(futures):
                status, payload = future.result()
                
                if status == "success":
                    # Save to disk immediately (append mode)
                    with open(output_file, "a", encoding="utf-8") as f:
                        f.write(json.dumps(payload) + "\n")
                    pbar.update(1)
                    
                    with results_lock:
                        results.append(payload)
                else:
                    with error_log_lock:
                        error_log.append(payload)

# 3. Save Logs
with open(os.getcwd() + "/data_for_git/fact_rating_log.txt", "w") as f:
    for error in error_log:
        f.write(error + "\n")

print(f"Processed {len(results)} responses successfully.")

                                                        
Rating Facts:  78%|███████▊  | 97108/124643 [00:00<00:00, 62411963.35it/s]

Starting from 3095/3413 (90.6826838558453%)


In [23]:
#write results to file with utf8 encoding
with open(os.getcwd() + "/data_for_git/rated_facts.jsonl", "w", encoding="utf-8") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")
